In [8]:
"""
Final cleaning pass on AllRoles_clean.csv, producing the dataset that's
actually ready to send to the NIOCCS API.

Output: one row per unique historical opportunity, restricted to the 4
core Career Launch/Spring Forward hubs, where both the role name and role
description have real (non-placeholder, non-blank) content. Short/generic
content is fine and is NOT filtered out here - only rows with nothing
usable are dropped.

FILTERS APPLIED (in order, each one printed so nothing disappears quietly)
---------------------------------------------------------------------------
1. Hub Normalized must be one of the 4 core hubs:
   Healthcare, STEM and Green, Community and Social Services,
   Marketing and Communications.
   (Government and Public Service is dropped here - it only ever existed
   in the 2022 cohort and isn't one of the 4 hubs going forward. Cultural
   Corps/I2E rows were already out of scope per the diagnostic pass.)
2. Opportunity Name (role name) must be non-blank.
3. Opportunity Description (role description) must be non-blank.
4. Opportunity Name must not be an unfilled placeholder/template stub
   (e.g. "Enter Your Job Title Here") - these have text, but it isn't
   real content, so they don't satisfy "at least something."
5. Duplicate Opportunity Ids are dropped, keeping the first occurrence
   (there weren't any in the raw file, but this makes the "one row per
   unique historical opportunity" guarantee explicit rather than assumed).

Everything that survives keeps its own text as-is, however short or
generic - per direction, that's fine.
"""

import pandas as pd
import re

IN_PATH = "../../data/AllRoles_clean.csv"
OUT_PATH = "../../data/AllRoles_nioccs_ready.csv"

CORE_HUBS = {
    "Healthcare",
    "STEM and Green",
    "Community and Social Services",
    "Marketing and Communications",
}

PLACEHOLDER_TITLE_RE = re.compile(
    r"enter your job title here|insert title of position here|"
    r"organization name.*job position|insert name of org",
    re.I,
)


def is_placeholder_title(title):
    if pd.isna(title):
        return False
    return bool(PLACEHOLDER_TITLE_RE.search(str(title)))


def main():
    df = pd.read_csv(IN_PATH, encoding="utf-8-sig", low_memory=False)
    n_start = len(df)
    print(f"Starting rows: {n_start}")

    # 1. Restrict to the 4 core hubs
    df = df[df["Hub Normalized"].isin(CORE_HUBS)]
    print(f"After restricting to the 4 core hubs: {len(df)} (-{n_start - len(df)})")
    n = len(df)

    # 2 & 3. Both title and description must be non-blank
    df = df[df["Opportunity Name"].notna() & (df["Opportunity Name"].str.strip() != "")]
    df = df[df["Opportunity Description"].notna() & (df["Opportunity Description"].str.strip() != "")]
    print(f"After requiring non-blank title AND description: {len(df)} (-{n - len(df)})")
    n = len(df)

    # 4. Drop unfilled placeholder/template postings
    df = df[~df["Opportunity Name"].apply(is_placeholder_title)]
    print(f"After dropping placeholder/template titles: {len(df)} (-{n - len(df)})")
    n = len(df)

    # 5. Guarantee uniqueness on Opportunity Id
    df = df.drop_duplicates(subset=["Opportunity Id"], keep="first")
    print(f"After deduplicating on Opportunity Id: {len(df)} (-{n - len(df)})")

    print(f"\nFinal dataset: {len(df)} rows ({len(df) / n_start * 100:.1f}% of starting rows)")
    print("\nRows per hub in final dataset:")
    print(df["Hub Normalized"].value_counts().to_string())
    print("\nRows per cohort year in final dataset:")
    print(df["Cohort Year"].value_counts().sort_index().to_string())

    # Keep a clean, descriptively-named set of columns for the API step.
    # "Hub Normalized" is what will be passed as i= for every row, including
    # STEM and Green - no special-casing there per direction.
    out = df[[
        "Opportunity Id", "Opportunity Name", "Opportunity Description",
        "Hub Normalized", "Cohort Year", "Program Family", "Agency Name",
    ]].rename(columns={
        "Opportunity Id": "Opportunity Identifier",
        "Opportunity Name": "Role Name",
        "Opportunity Description": "Role Description",
        "Hub Normalized": "Program Hub Category",
    })

    out.to_csv(OUT_PATH, index=False)
    print(f"\nWrote {OUT_PATH}")

    return out


if __name__ == "__main__":
    main()

Starting rows: 9297
After restricting to the 4 core hubs: 7194 (-2103)
After requiring non-blank title AND description: 7193 (-1)
After dropping placeholder/template titles: 7125 (-68)
After deduplicating on Opportunity Id: 7125 (-0)

Final dataset: 7125 rows (76.6% of starting rows)

Rows per hub in final dataset:
Hub Normalized
Community and Social Services    2398
Marketing and Communications     1963
STEM and Green                   1445
Healthcare                       1319

Rows per cohort year in final dataset:
Cohort Year
2022.0     603
2023.0    1416
2024.0    1580
2025.0    1673
2026.0    1853

Wrote ../../data/AllRoles_nioccs_ready.csv


In [9]:
# --- NIOCCS NAICS/SOC auto-classification - call the API on AllRoles_nioccs_ready.csv ---


import pandas as pd
import requests
import time
import json
import os

IN_PATH = "../../data/AllRoles_nioccs_ready.csv"
CHECKPOINT_PATH = "../../data/AllRoles_nioccs_coded.csv"

NIOCCS_BASE_URL = "https://wwwn.cdc.gov/nioccs/IOCode"
REQUEST_PARAMS_STATIC = {"c": 2, "v": 18, "u": 1, "n": 1}

# Cap how much of Role Description gets sent - full descriptions run to
# several thousand characters (boilerplate WORK SCHEDULE/LOCATION sections
# included), and very long query strings risk hitting URL-length limits on
# a GET request. 800 characters comfortably covers the WHO WE ARE / DUTIES
# & RESPONSIBILITIES sections for most postings - adjust if you find it's
# cutting off useful content.
MAX_DESCRIPTION_CHARS = 800

# TEST_MODE = True runs on just the first TEST_SAMPLE_SIZE rows and prints
# the raw response for each, so you can sanity-check field names before
# committing to the full batch. Flip to False (and re-run the cell) once
# you've confirmed the parsing looks right.
TEST_MODE = True
TEST_SAMPLE_SIZE = 10

# Be a polite, unauthenticated guest on a free public CDC endpoint.
SECONDS_BETWEEN_REQUESTS = 0.5
MAX_RETRIES_PER_ROW = 3

# Sentinel codes per the NIOCCS spec - these mean "insufficient information
# to code," not a real match.
NAICS_INSUFFICIENT_INFO_CODE = "009990"
SOC_INSUFFICIENT_INFO_CODE = "00-9900"


def build_occupation_text(role_name: str, role_description: str) -> str:
    role_name = "" if pd.isna(role_name) else str(role_name).strip()
    role_description = "" if pd.isna(role_description) else str(role_description).strip()
    combined = f"{role_name}. {role_description}"
    return combined[:MAX_DESCRIPTION_CHARS]


def call_nioccs(industry_text: str, occupation_text: str) -> dict:
    """
    Calls the NIOCCS API for a single row. Returns a dict with:
      - 'raw_response': the parsed JSON (or None if the call failed)
      - 'http_status': status code (or None)
      - 'error': error message if something went wrong, else None
    Retries transient failures a few times before giving up on the row.
    """
    params = {
        **REQUEST_PARAMS_STATIC,
        "i": industry_text,
        "o": occupation_text,
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        try:
            resp = requests.get(NIOCCS_BASE_URL, params=params, timeout=20)
            if resp.status_code == 200:
                try:
                    return {"raw_response": resp.json(), "http_status": 200, "error": None}
                except json.JSONDecodeError:
                    last_error = f"Non-JSON response (status 200): {resp.text[:200]}"
            else:
                last_error = f"HTTP {resp.status_code}: {resp.text[:200]}"
        except requests.exceptions.RequestException as e:
            last_error = f"Request exception: {e}"

        if attempt < MAX_RETRIES_PER_ROW:
            time.sleep(1.5 * attempt)  # simple backoff before retrying

    return {"raw_response": None, "http_status": None, "error": last_error}


def parse_nioccs_response(raw) -> dict:
    """
    Best-effort extraction of the fields the spec describes: matched NAICS
    code/title/probability, matched SOC code/title/probability, and the
    insufficient-information / implausible-pairing flags.

    THIS IS UNVERIFIED against a real response - print raw_response for
    your first few rows and adjust the keys below (`.get(...)` calls) to
    match whatever NIOCCS actually returns. Every lookup is guarded so a
    wrong key just yields None/blank rather than crashing the loop.
    """
    empty = {
        "naics_code": None, "naics_title": None, "naics_probability": None,
        "soc_code": None, "soc_title": None, "soc_probability": None,
        "naics_insufficient_info": None, "soc_insufficient_info": None,
        "implausible_pairing_flag": None,
    }
    if raw is None:
        return empty

    # NIOCCS sometimes wraps results in a list - handle both shapes.
    record = raw[0] if isinstance(raw, list) and raw else raw
    if not isinstance(record, dict):
        return empty

    naics_code = record.get("NAICS") or record.get("Naics_Cd") or record.get("naics_code")
    soc_code = record.get("SOC") or record.get("Soc_Cd") or record.get("soc_code")

    return {
        "naics_code": naics_code,
        "naics_title": record.get("NAICS_Title") or record.get("Naics_Title"),
        "naics_probability": record.get("NAICS_Prob") or record.get("Naics_Prob"),
        "soc_code": soc_code,
        "soc_title": record.get("SOC_Title") or record.get("Soc_Title"),
        "soc_probability": record.get("SOC_Prob") or record.get("Soc_Prob"),
        "naics_insufficient_info": (naics_code == NAICS_INSUFFICIENT_INFO_CODE) if naics_code else None,
        "soc_insufficient_info": (soc_code == SOC_INSUFFICIENT_INFO_CODE) if soc_code else None,
        "implausible_pairing_flag": record.get("Flag_Implausible") or record.get("implausible"),
    }


def load_already_coded():
    """Resume support: skip rows we've already successfully coded."""
    if os.path.exists(CHECKPOINT_PATH):
        done = pd.read_csv(CHECKPOINT_PATH)
        return done, set(done["Opportunity Identifier"])
    return pd.DataFrame(), set()


def run():
    df = pd.read_csv(IN_PATH, encoding="utf-8-sig")

    if TEST_MODE:
        df = df.head(TEST_SAMPLE_SIZE)
        print(f"TEST MODE: running on {len(df)} rows only. Set TEST_MODE = False for the full batch.\n")

    done_df, done_ids = load_already_coded()
    if not TEST_MODE and done_ids:
        print(f"Resuming: {len(done_ids)} rows already coded in a previous run, skipping those.")
        df = df[~df["Opportunity Identifier"].isin(done_ids)]

    results = []
    total = len(df)
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        opportunity_id = row["Opportunity Identifier"]
        role_name = row["Role Name"]
        role_description = row["Role Description"]
        industry_text = row["Program Hub Category"]  # STEM and Green included, no special-casing

        occupation_text = build_occupation_text(role_name, role_description)
        api_result = call_nioccs(industry_text, occupation_text)
        parsed = parse_nioccs_response(api_result["raw_response"])

        result_row = {
            "Opportunity Identifier": opportunity_id,
            "Role Name": role_name,
            "Program Hub Category": industry_text,
            "NAICS Code": parsed["naics_code"],
            "NAICS Title": parsed["naics_title"],
            "NAICS Match Probability": parsed["naics_probability"],
            "SOC Code": parsed["soc_code"],
            "SOC Title": parsed["soc_title"],
            "SOC Match Probability": parsed["soc_probability"],
            "NAICS Flagged As Insufficient Information": parsed["naics_insufficient_info"],
            "SOC Flagged As Insufficient Information": parsed["soc_insufficient_info"],
            "Flagged As Implausible NAICS/SOC Pairing": parsed["implausible_pairing_flag"],
            "API Call Error": api_result["error"],
        }
        results.append(result_row)

        if TEST_MODE:
            print(f"--- Row {i}/{total}: {role_name!r} ---")
            print("Raw response:", json.dumps(api_result["raw_response"], indent=2)[:1000])
            print("Parsed:", {k: v for k, v in result_row.items() if k not in ("Opportunity Identifier", "Role Name")})
            print()
        elif i % 25 == 0:
            print(f"...{i}/{total} rows called")

        time.sleep(SECONDS_BETWEEN_REQUESTS)

        # Checkpoint every 100 rows in full-batch mode, so a crash mid-run
        # doesn't cost you the whole batch.
        if not TEST_MODE and i % 100 == 0:
            combined = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            combined.to_csv(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved: {len(combined)} rows coded so far.")

    results_df = pd.DataFrame(results)

    if not TEST_MODE:
        combined = pd.concat([done_df, results_df], ignore_index=True)
        combined.to_csv(CHECKPOINT_PATH, index=False)
        print(f"\nDone. {len(combined)} total rows coded, written to {CHECKPOINT_PATH}")
        n_errors = combined["API Call Error"].notna().sum()
        if n_errors:
            print(f"{n_errors} rows had a call error and should be re-run separately - "
                  f"they were still saved to the checkpoint file with the error message.")
    else:
        print(f"\nTest run complete on {len(results_df)} rows. Nothing written to disk yet - "
              f"review the raw responses above, fix parse_nioccs_response if the keys don't "
              f"match, then set TEST_MODE = False and re-run this cell for the full batch.")

    return results_df


results_df = run()
results_df

TEST MODE: running on 10 rows only. Set TEST_MODE = False for the full batch.

--- Row 1/10: 'Dental Assistant - Open Bright Pediatric Dentistry' ---
Raw response: {
  "Industry": [
    {
      "NAICSCode": "621210",
      "NAICSTitle": "Offices of Dentists",
      "NAICSProbability": 0.609683335,
      "CensusIndustryCode": "7980",
      "CensusIndustryTitle": "Offices of Dentists"
    }
  ],
  "Occupation": [
    {
      "SOCCode": "29-1292",
      "SOCTitle": "Dental Hygienists",
      "SOCProbability": 0.621906042,
      "CensusOccupationCode": "3310",
      "CensusOccupationTitle": "Dental Hygienists"
    }
  ],
  "UnexpectedCodeCombination": "N",
  "Scheme": "NAICS 2017 and SOC 2018, Census Industry and Occupation 2018"
}
Parsed: {'Program Hub Category': 'Healthcare', 'NAICS Code': None, 'NAICS Title': None, 'NAICS Match Probability': None, 'SOC Code': None, 'SOC Title': None, 'SOC Match Probability': None, 'NAICS Flagged As Insufficient Information': None, 'SOC Flagged As Insuff

,Opportunity Identifier,Role Name,Program Hub Category,NAICS Code,NAICS Title,NAICS Match Probability,SOC Code,SOC Title,SOC Match Probability,NAICS Flagged As Insufficient Information,SOC Flagged As Insufficient Information,Flagged As Implausible NAICS/SOC Pairing,API Call Error
0,395bf964-a1f1-455f-9e41-08dd7b582abd,Dental Assistant - Open Bright Pediatric Denti...,Healthcare,None,None,None,None,None,None,None,None,None,None
1,23174e7a-34b2-4373-9e42-08dd7b582abd,Dental receptionist - Open Bright Pediatric De...,Healthcare,None,None,None,None,None,None,None,None,None,None
2,e303552d-c68a-4108-9c75-08dd7c328786,Finance/Accounting Intern – PEN America,Community and Social Services,None,None,None,None,None,None,None,None,None,None
3,456bc9db-15c3-4d6f-e196-08dd81aa123b,Software Engineer Intern - Icahn School of Med...,Healthcare,None,None,None,None,None,None,None,None,None,None
4,aa75f149-6cb8-40e8-e197-08dd81aa123b,Website Intern - Icahn School of Medicine at M...,Healthcare,None,None,None,None,None,None,None,None,None,None
5,cffbac57-aab7-4ef7-e198-08dd81aa123b,Digital Marketing Intern - Mount Sinai Health ...,Healthcare,None,None,None,None,None,None,None,None,None,None
6,87e69a26-73af-403e-9ce2-08dd7c328786,Office Assistant - Cobble Hill Health Center,Healthcare,None,None,None,None,None,None,None,None,None,None
7,a34fabed-ccca-4cb2-9ce3-08dd7c328786,Training & Technical Assistance Partnerships I...,Community and Social Services,None,None,None,None,None,None,None,None,None,None
8,50b492c2-8ae5-4b17-e182-08dd81aa123b,Lectec - Graphic Designer / Content Creator In...,STEM and Green,None,None,None,None,None,None,None,None,None,None
9,1cd698ee-e79f-4be1-e17f-08dd81aa123b,Lectec - Content Producer / Videographer Intern,STEM and Green,None,None,None,None,None,None,None,None,None,None
